# PPO vs CISPO in PyTorch

One prompt, one written token, a two-layer network. Autograd computes every gradient; the only thing that
differs between the losses is the factor `w` in front of `A * (p - e_y)` on the logits.

- PPO: `w = r`, or `0` when clipped. The clip makes the loss a constant, so autograd returns 0.
- CISPO: `w = clamp(r, 0, 4).detach()`. A frozen number, never 0 unless `r` is 0.

In [1]:
import numpy as np
import torch
torch.set_printoptions(precision=3, sci_mode=False)

TOKENS = ["So", "The", "Wait"]                    # vocabulary, V = 3
x = torch.tensor([1.0, 0.5], dtype=torch.float64) # (2,)   the prompt

rng = np.random.default_rng(4)
W1_init = torch.tensor(rng.normal(0, 1, (2, 3))) # (2, 3) input -> hidden
W2_init = torch.tensor(rng.normal(0, 1, (3, 3))) # (3, 3) hidden -> logits


def forward(W1, W2):
    """W1: (2, 3), W2: (3, 3) -> logits z: (3,), log-probs log_p: (3,)"""
    h = torch.tanh(W1.T @ x)                      # (3,)  hidden
    z = W2.T @ h                                  # (3,)  logits: one score per token
    log_p = torch.log_softmax(z, dim=0)           # (3,)  log-probabilities
    return z, log_p

In [2]:
# Rollout: the network writes "Wait". Record its probability now. No gradient: it is data from here on.
with torch.no_grad():
    _, log_p = forward(W1_init, W2_init)          # (3,)
y = 2                                             # index of "Wait"
log_p_old = log_p[y].item()                       # ()  log 0.045, fixed forever
A = 1.0                                           # ()  advantage: the response was good, push "Wait" up
print("p at rollout:", log_p.exp(), "  p_old:", round(np.exp(log_p_old), 3))

p at rollout: tensor([0.491, 0.464, 0.045], dtype=torch.float64)   p_old: 0.045


In [3]:
def losses(log_p, kind):
    """log_p: (3,) with gradient -> loss: () and the factor w: () (for printing)."""
    log_p_y = log_p[y]                            # ()  log-prob of the written token, has gradient
    r = torch.exp(log_p_y - log_p_old)            # ()  ratio p_now / p_old, has gradient

    if kind == "reinforce":
        loss = -A * log_p_y                       # ()  plain policy gradient
        w = 1.0

    elif kind == "ppo":
        # loss = -min(r * A, clamp(r) * A).
        # Inside [0.8, 1.28], clamp(r) = r and the gradient flows through r.
        # Outside, clamp(r) is the constant 0.8 or 1.28; if the min picks it, the loss is a constant -> gradient 0.
        r_clip = torch.clamp(r, 0.8, 1.28)        # ()
        loss = -torch.min(r * A, r_clip * A)      # ()
        w = r.item() if (r * A <= r_clip * A) else 0.0

    elif kind == "cispo":
        # loss = -w * A * log p_y with w = clamp(r, 0, 4) frozen.
        # .detach() turns w into a plain number: autograd differentiates only log_p_y.
        w_t = torch.clamp(r, 0.0, 4.0).detach()   # ()  miles tinker_losses.py:211
        loss = -w_t * A * log_p_y                 # ()  miles tinker_losses.py:212
        w = w_t.item()

    return loss, w

In [4]:
# Gradient on the logits, from autograd, vs w * A * (p - e_y).
# Step 0: weights = rollout weights, r = 1, all three agree.
# Step 1: weights after one REINFORCE-sized step, r = 1.516: PPO gives 0, CISPO gives 1.516 * (p - e_y).
def check(W1_start, W2_start, kind):
    W1 = W1_start.detach().clone().requires_grad_()   # (2, 3)
    W2 = W2_start.detach().clone().requires_grad_()   # (3, 3)
    z, log_p = forward(W1, W2)                    # (3,), (3,)
    z.retain_grad()                               # keep dloss/dz so we can print it
    loss, w = losses(log_p, kind)
    loss.backward()

    p = log_p.exp().detach()                      # (3,)
    e_y = torch.nn.functional.one_hot(torch.tensor(y), 3).double()   # (3,)  [0, 0, 1]
    print(f"  {kind:9s}  w = {w:.3f}   dz = {z.grad}   w*A*(p - e_y) = {w * A * (p - e_y)}")

In [5]:
# 8 optimizer steps on the same rollout. log_p_old stays fixed; the weights move, so r and w change.
def train(kind, steps=8, lr=0.1):
    W1 = W1_init.clone().requires_grad_()         # (2, 3)
    W2 = W2_init.clone().requires_grad_()         # (3, 3)
    opt = torch.optim.SGD([W1, W2], lr=lr)        # W <- W - lr * dW
    rows = []
    for step in range(steps):
        _, log_p = forward(W1, W2)
        loss, w = losses(log_p, kind)
        rows.append((step, log_p[y].exp().item(), np.exp(log_p[y].item() - log_p_old), w))
        opt.zero_grad()
        loss.backward()
        opt.step()
    return W1, W2, rows

for kind in ["ppo", "cispo"]:
    *_, rows = train(kind)
    print(f"{kind.upper():6s} step  p_y     r       w")
    for step, p_y, r, w in rows:
        print(f"       {step}     {p_y:.3f}  {r:6.3f}  {w:.3f}")
    print()

# PPO:   step 1, r = 1.516 > 1.28 with A > 0: the min picks the constant 1.28 * A, every gradient is 0,
#        and the weights never move again.
# CISPO: w = r until r passes 4 at step 3, then w = 4. "Wait" keeps rising.

PPO    step  p_y     r       w
       0     0.045   1.000  1.000
       1     0.068   1.516  0.000
       2     0.068   1.516  0.000
       3     0.068   1.516  0.000
       4     0.068   1.516  0.000
       5     0.068   1.516  0.000
       6     0.068   1.516  0.000
       7     0.068   1.516  0.000

CISPO  step  p_y     r       w
       0     0.045   1.000  1.000
       1     0.068   1.516  1.516
       2     0.119   2.650  2.650
       3     0.257   5.713  4.000
       4     0.502  11.168  4.000
       5     0.657  14.610  4.000
       6     0.743  16.536  4.000
       7     0.796  17.711  4.000



In [6]:
W1_s1, W2_s1, _ = train("cispo", steps=1)         # weights after step 0 (same for all three: w = 1 at step 0)
for name, (W1_s, W2_s) in [("step 0", (W1_init, W2_init)), ("step 1", (W1_s1, W2_s1))]:
    print(name)
    for kind in ["reinforce", "ppo", "cispo"]:
        check(W1_s, W2_s, kind)

step 0
  reinforce  w = 1.000   dz = tensor([ 0.491,  0.464, -0.955], dtype=torch.float64)   w*A*(p - e_y) = tensor([ 0.491,  0.464, -0.955], dtype=torch.float64)
  ppo        w = 1.000   dz = tensor([ 0.491,  0.464, -0.955], dtype=torch.float64)   w*A*(p - e_y) = tensor([ 0.491,  0.464, -0.955], dtype=torch.float64)
  cispo      w = 1.000   dz = tensor([ 0.491,  0.464, -0.955], dtype=torch.float64)   w*A*(p - e_y) = tensor([ 0.491,  0.464, -0.955], dtype=torch.float64)
step 1
  reinforce  w = 1.000   dz = tensor([ 0.501,  0.431, -0.932], dtype=torch.float64)   w*A*(p - e_y) = tensor([ 0.501,  0.431, -0.932], dtype=torch.float64)
  ppo        w = 0.000   dz = tensor([0., 0., 0.], dtype=torch.float64)   w*A*(p - e_y) = tensor([0., 0., -0.], dtype=torch.float64)
  cispo      w = 1.516   dz = tensor([ 0.759,  0.654, -1.413], dtype=torch.float64)   w*A*(p - e_y) = tensor([ 0.759,  0.654, -1.413], dtype=torch.float64)


In [7]:
# Why .detach() matters: without it, autograd also differentiates w = r, which depends on the weights.
# The extra term flips the push at step 1: "Wait" would go DOWN even though A > 0.
W1, W2, _ = train("cispo", steps=1)               # weights after one step: r = 1.516
for detach in [True, False]:
    W1_ = W1.detach().clone().requires_grad_()
    W2_ = W2.detach().clone().requires_grad_()
    z, log_p = forward(W1_, W2_)
    z.retain_grad()
    r = torch.exp(log_p[y] - log_p_old)           # ()
    w_t = torch.clamp(r, 0.0, 4.0)
    w_t = w_t.detach() if detach else w_t
    (-w_t * A * log_p[y]).backward()
    print(f"detach={detach!s:5s}  dz for 'Wait' = {z.grad[y]:+.3f}   ({'up' if z.grad[y] < 0 else 'DOWN'})")

detach=True   dz for 'Wait' = -1.413   (up)
detach=False  dz for 'Wait' = +2.382   (DOWN)


In our runs there is one optimizer step per rollout, so only step 0 happens: `r = 1`, and REINFORCE, PPO and
CISPO give the same gradient.